# 개념 1. 외삽과 외삽 공식

**목표:** 외삽이 무엇인지 알고, 외삽을 처음 배울 때 쓰는 **선형 외삽 공식**으로 직접 계산해 본다.

순서
1. 외삽이란
2. 외삽 공식 (선형 외삽)
3. 예시: 휴대폰 배터리
4. 예시: 택시 요금
5. 일상에서 어디에 쓰이나
6. 참고: 가장 단순하게는 마지막 값을 그대로
7. 정리

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"   # 한글 글꼴 (macOS)
plt.rcParams["axes.unicode_minus"] = False

## 1. 외삽이란

어떤 값을 몇 번 쟀다고 하자.

- 잰 점들 **사이**의 값을 추정하면 **보간**(내삽)
- 잰 범위 **밖**의 값을 추정하면 **외삽**

```
        잰 범위
  ●─────●─────●  ?  ?  ?
  ↑ 보간: 사이   └ 외삽: 밖
```

| | 보간 | 외삽 |
|---|---|---|
| 추정하는 곳 | 잰 점들 사이 | 잰 범위 밖 (보통 미래) |
| 붙잡아 주는 것 | 양쪽 점 | 한쪽 점들뿐 |
| 믿을 만한 정도 | 높다 | 멀어질수록 낮다 |

외삽은 반드시 **가정**을 깔고 한다: *"지금까지의 흐름이 앞으로도 이어진다."*

## 2. 외삽 공식 (선형 외삽)

외삽을 처음 배울 때 쓰는 공식이다. 가정은 *"같은 비율로 계속 변한다"*.

두 점 $(x_1, y_1)$, $(x_2, y_2)$를 알면, 범위 밖 $x$에서의 값은

$$y = y_2 + \underbrace{\frac{y_2 - y_1}{x_2 - x_1}}_{\text{기울기}} \times (x - x_2)$$

두 점을 잇는 직선을 범위 밖까지 늘린 것이다.

- **기울기**: $x$가 1 늘 때 $y$가 얼마나 변하나
- **$(x - x_2)$**: 마지막 점에서 얼마나 멀리 가나

In [ ]:
def linear_extrapolate(x, x1, y1, x2, y2):
    """두 점 (x1, y1), (x2, y2)를 지나는 직선의 x 에서의 값 (선형 외삽)."""
    slope = (y2 - y1) / (x2 - x1)
    return y2 + slope * (x - x2)

## 3. 예시: 휴대폰 배터리

10시에 80%, 11시에 70%였다. 14시에는 몇 %일까? 몇 시에 꺼질까?

In [ ]:
x1, y1, x2, y2 = 10, 80, 11, 70
slope = (y2 - y1) / (x2 - x1)
print("기울기:", slope, "%/시간")
print("14시 예상:", linear_extrapolate(14, x1, y1, x2, y2), "%")
print("0%가 되는 시각:", x2 + (0 - y2) / slope, "시")

In [ ]:
hours = np.linspace(11, 18.5, 50)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(hours, linear_extrapolate(hours, x1, y1, x2, y2), "--", label="외삽 (시간당 −10%)")
ax.plot([x1, x2], [y1, y2], "o", color="k", ms=8, label="잰 값")
ax.plot(14, linear_extrapolate(14, x1, y1, x2, y2), "s", color="C3", label="14시 예상")
ax.axhline(0, color="gray", lw=1)
ax.set_xlabel("시각"); ax.set_ylabel("배터리 (%)"); ax.set_ylim(-5, 90)
ax.grid(alpha=.3); ax.legend(); plt.show()

## 4. 예시: 택시 요금

외삽의 $x$가 꼭 시간일 필요는 없다. 2km에 4,800원, 4km에 7,200원 나왔다. 10km면?

In [ ]:
fare = linear_extrapolate(10, 2, 4800, 4, 7200)
print(f"km당 {(7200 - 4800) / (4 - 2):.0f}원, 10km 예상 {fare:,.0f}원")

## 5. 일상에서 어디에 쓰이나

모르는 사이에 매일 외삽 결과를 보고 있다. 대부분 *"지금 속도가 이어진다"*는 선형 외삽이다.

| 어디서 | 아는 것 (잰 값) | 외삽하는 것 |
|---|---|---|
| 휴대폰 배터리 "약 3시간 남음" | 최근 배터리가 줄어드는 속도 | 0%가 되는 시각 |
| 파일 다운로드 "2분 남음" | 지금까지 받은 양과 속도 | 다 받는 시각 |
| 내비게이션 도착 예정 시간 | 지금 위치와 속도 | 도착 시각 |
| 지도 앱의 내 위치 (터널 안) | GPS가 끊기기 직전 위치와 속도 | 지금 위치 |
| 게임의 상대 캐릭터 움직임 | 마지막으로 받은 위치와 속도 | 다음 신호까지의 위치 |
| 주가 추세선, 인구 예측 | 지난 몇 년의 값 | 앞으로의 값 |

터널 속 내 위치, 게임 캐릭터는 **"마지막 위치 + 속도 × 지난 시간"**으로 위치를 미는 것이다. 배가 AIS 보고 사이에 화면에서 움직이는 것도 똑같은 원리다(해양 응용 1).

In [ ]:
# 다운로드 예시: 30초 동안 120MB 받았고, 파일은 700MB
done, elapsed, total = 120, 30, 700
speed = done / elapsed
print(f"속도 {speed:.1f} MB/초, 남은 시간 {(total - done) / speed:.0f}초")

# 터널 예시: GPS가 끊길 때 0m 지점, 시속 90km. 20초 뒤 위치는?
v = 90 * 1000 / 3600
print(f"20초 뒤 터널 입구에서 {v * 20:.0f} m")

## 6. 참고: 가장 단순하게는 마지막 값을 그대로

공식 없이 *"앞으로도 마지막 값과 같다"*고 보는 방법도 있다(**0차 외삽**이라고도 한다). 배터리라면 "14시에도 70%"라고 보는 것이다.

아무것도 예상하지 않는 것과 같아서, 변하는 값에는 바로 틀린다. 그래서 외삽이라고 하면 보통 **선형 외삽**을 말한다.

## 직접 해 보기

1. 배터리가 11시에 70%가 아니라 72%였다면 몇 시에 꺼지나?
2. 택시 요금으로 1km 요금을 거꾸로 추정하면 얼마인가? 이것도 외삽인가?

In [ ]:
# 여기서 직접 바꿔 보세요


## 정리

- 외삽: 잰 범위 **밖**을 **가정**으로 추정한다.
- 선형 외삽 공식: $y = y_2 + \text{기울기} \times (x - x_2)$. 두 점을 잇는 직선을 늘린다.

**다음 (개념 2):** 선형 외삽은 언제 틀리고, 어떻게 보완하나.